# 08 - Reply Generation

Generate human-reviewable support replies using ML predictions, multilingual RAG retrieval, and English/German-aware templates.

Important design:
- ML predictions are shown as internal decision-support metadata.
- Customer-facing replies are policy-led and avoid exposing raw labels like `Incident`, `Problem`, or predicted queue.
- Optional LLM generation is supported in `src/generate_reply.py`, but it is off by default to avoid API cost.


In [1]:
import os
import sys
import pandas as pd

sys.path.append("..")

In [2]:
from src.generate_reply import (
    analyze_ticket_and_generate_reply,
    detect_language_safe,
    load_models,
)

In [3]:
models = load_models()
print("All models loaded successfully.")

All models loaded successfully.


## English/German Language Detection Check

In [4]:
sample_tickets = [
    "My laptop arrived damaged and I want a refund.",
    "Ich habe einen beschädigten Laptop erhalten und möchte eine Rückerstattung.",
]

for ticket in sample_tickets:
    print(ticket)
    print("Detected language:", detect_language_safe(ticket))
    print("-" * 80)

My laptop arrived damaged and I want a refund.
Detected language: en
--------------------------------------------------------------------------------
Ich habe einen beschädigten Laptop erhalten und möchte eine Rückerstattung.
Detected language: de
--------------------------------------------------------------------------------


## Single End-to-End Test

In [5]:
output = analyze_ticket_and_generate_reply(
    "My charger stopped working after 6 months and I need warranty support.",
    use_llm=False,
)

output

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'ticket': 'My charger stopped working after 6 months and I need warranty support.',
 'language': 'en',
 'model_predictions': {'ticket_type': 'Incident',
  'queue': 'IT Support',
  'priority': 'high',
  'sentiment': 'negative'},
 'policy_source': 'warranty_policy.txt',
 'policy_context': '[Policy Source 1: warranty_policy.txt]\nWarranty Policy\n\nProducts include a 1-year limited warranty from the date of purchase.\n\nWarranty covers:\n- Manufacturing defects.\n- Hardware failure under normal use.\n- Battery or charging defects.\n- Product malfunction not caused by misuse.\n- Device failure within the warranty period.\n\nWarranty does not cover:\n- Physical damage caused by the customer.\n- Water damage.\n- Unauthorized repair.\n- Accidental damage.\n- Damage after warranty expiry.\n\n[Policy Source 2: refund_policy.txt]\nRefunds are not allowed when:\n- The product was damaged because of customer misuse.\n- The refund request is made after 30 days.\n- The customer cannot provide proof

In [6]:
german_output = analyze_ticket_and_generate_reply(
    "Mein Ladegerät funktioniert nach 6 Monaten nicht mehr und ich brauche Garantieunterstützung.",
    use_llm=False,
)

german_output

{'ticket': 'Mein Ladegerät funktioniert nach 6 Monaten nicht mehr und ich brauche Garantieunterstützung.',
 'language': 'de',
 'model_predictions': {'ticket_type': 'Request',
  'queue': 'IT Support',
  'priority': 'high',
  'sentiment': 'neutral'},
 'policy_source': 'warranty_policy.txt',
 'policy_context': '[Policy Source 1: warranty_policy.txt]\nWarranty Policy\n\nProducts include a 1-year limited warranty from the date of purchase.\n\nWarranty covers:\n- Manufacturing defects.\n- Hardware failure under normal use.\n- Battery or charging defects.\n- Product malfunction not caused by misuse.\n- Device failure within the warranty period.\n\nWarranty does not cover:\n- Physical damage caused by the customer.\n- Water damage.\n- Unauthorized repair.\n- Accidental damage.\n- Damage after warranty expiry.\n\n[Policy Source 2: refund_policy.txt]\nRefunds are not allowed when:\n- The product was damaged because of customer misuse.\n- The refund request is made after 30 days.\n- The customer 

## Batch Reply Examples

In [7]:
test_tickets = [
    "My laptop arrived damaged and I want a refund.",
    "Ich habe einen beschädigten Laptop erhalten und möchte eine Rückerstattung.",
    "My order has not arrived and tracking is not updating.",
    "Mein Paket ist noch nicht angekommen und die Sendungsverfolgung aktualisiert sich nicht.",
    "My phone battery drains very quickly.",
    "Der Akku meines Telefons entlädt sich sehr schnell.",
    "I cannot login to my account.",
    "Ich kann mich nicht in mein Konto einloggen.",
]

generated_results = []

for ticket in test_tickets:
    result = analyze_ticket_and_generate_reply(ticket, use_llm=False)
    generated_results.append(result)

generated_df = pd.DataFrame(generated_results)

display_columns = [
    "ticket",
    "language",
    "model_predictions",
    "policy_source",
    "policy_suggested_queue",
    "policy_suggested_action",
    "generation_mode",
    "generated_reply",
]

generated_df[display_columns]


,ticket,language,model_predictions,policy_source,policy_suggested_queue,policy_suggested_action,generation_mode,generated_reply
0,My laptop arrived damaged and I want a refund.,en,"{'ticket_type': 'Incident', 'queue': 'Technica...",refund_policy.txt,Billing or Customer Support,review refund eligibility and ask for proof of...,template,"Dear Customer,\n\nWe’re sorry to hear about th..."
1,Ich habe einen beschädigten Laptop erhalten un...,de,"{'ticket_type': 'Change', 'queue': 'Returns an...",refund_policy.txt,Billing or Customer Support,review refund eligibility and ask for proof of...,template,"Sehr geehrte Kundin, sehr geehrter Kunde,\n\nV..."
2,My order has not arrived and tracking is not u...,en,"{'ticket_type': 'Problem', 'queue': 'Technical...",shipping_policy.txt,Logistics or Customer Support,check tracking status and verify courier infor...,template,"Dear Customer,\n\nWe’re sorry to hear about th..."
3,Mein Paket ist noch nicht angekommen und die S...,de,"{'ticket_type': 'Problem', 'queue': 'Customer ...",shipping_policy.txt,Logistics or Customer Support,check tracking status and verify courier infor...,template,"Sehr geehrte Kundin, sehr geehrter Kunde,\n\nV..."
4,My phone battery drains very quickly.,en,"{'ticket_type': 'Incident', 'queue': 'Customer...",warranty_policy.txt,Warranty Support or Technical Support,verify purchase date and warranty eligibility,template,"Dear Customer,\n\nThank you for reaching out t..."
5,Der Akku meines Telefons entlädt sich sehr sch...,de,"{'ticket_type': 'Problem', 'queue': 'Technical...",warranty_policy.txt,Warranty Support or Technical Support,verify purchase date and warranty eligibility,template,"Sehr geehrte Kundin, sehr geehrter Kunde,\n\nV..."
6,I cannot login to my account.,en,"{'ticket_type': 'Incident', 'queue': 'Technica...",account_policy.txt,Account Support,verify identity and check account access status,template,"Dear Customer,\n\nWe’re sorry to hear about th..."
7,Ich kann mich nicht in mein Konto einloggen.,de,"{'ticket_type': 'Incident', 'queue': 'Billing ...",account_policy.txt,Account Support,verify identity and check account access status,template,"Sehr geehrte Kundin, sehr geehrter Kunde,\n\nV..."


## Inspect Generated Replies

In [8]:
for _, row in generated_df.iterrows():
    print("=" * 100)
    print("Ticket:", row["ticket"])
    print("Language:", row["language"])
    print("Model Predictions:", row["model_predictions"])
    print("Policy Source:", row["policy_source"])
    print("Policy Suggested Queue:", row["policy_suggested_queue"])
    print("Policy Suggested Action:", row["policy_suggested_action"])
    print("Generation Mode:", row["generation_mode"])
    print("\nGenerated Reply:\n")
    print(row["generated_reply"])
    print()


Ticket: My laptop arrived damaged and I want a refund.
Language: en
Model Predictions: {'ticket_type': 'Incident', 'queue': 'Technical Support', 'priority': 'high', 'sentiment': 'negative'}
Policy Source: refund_policy.txt
Policy Suggested Queue: Billing or Customer Support
Policy Suggested Action: review refund eligibility and ask for proof of purchase
Generation Mode: template

Generated Reply:

Dear Customer,

We’re sorry to hear about the issue you’re facing.

Based on your message, we found a relevant company policy related to this issue.

We understand this may need prompt attention, so the support team should review it carefully.

Relevant policy reference:
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests sho

## Save Reply Examples

In [9]:
os.makedirs("../reports", exist_ok=True)

REPORT_PATH = "../reports/generated_reply_examples_multilingual.csv"
generated_df.to_csv(REPORT_PATH, index=False)

print("Saved generated replies to:", REPORT_PATH)


Saved generated replies to: ../reports/generated_reply_examples_multilingual.csv


## Optional LLM Test

Only run this after creating `.env` with `OPENAI_API_KEY`. Keep `use_llm=False` for normal free/offline testing. This cell is intentionally commented out because notebooks that accidentally spend money are a uniquely modern clown show.

In [10]:
# llm_output = analyze_ticket_and_generate_reply(
#     "Ich habe ein beschädigtes Produkt erhalten und möchte eine Rückerstattung.",
#     use_llm=True,
# )
# print(llm_output["generated_reply"])